In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from scipy.spatial.distance import euclidean
from sklearn.metrics.pairwise import cosine_similarity

# 1) 데이터 불러오기
df = pd.read_excel('/mnt/experiment1_analysis_overall.xlsx')

In [3]:
# 방어형 = "defensive", 무사과 = "no_apology" 라고 가정
df = df[df['apology_type'].isin(['defensive', 'no_apology'])]

In [5]:
# ---------------------------
# 분석 1) OLS/GLM (개별 DV)
# ---------------------------

dv_vars = ['Manip_ratio_neg', 'Avoid_ratio_neg', 'Excuse_ratio_neg',
            'Disrespect_ratio_neg', 'Insinc_ratio_neg']

results = {}

for dv in dv_vars:
    model = smf.ols(f"{dv} ~ C(apology_type)", data=df).fit()
    results[dv] = model
    print(f"\n=== {dv} 결과 ===")
    print(model.summary())


=== Manip_ratio_neg 결과 ===
                            OLS Regression Results                            
Dep. Variable:        Manip_ratio_neg   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     2.326
Date:                Fri, 16 Jan 2026   Prob (F-statistic):              0.130
Time:                        06:52:04   Log-Likelihood:                 43.344
No. Observations:                 118   AIC:                            -82.69
Df Residuals:                     116   BIC:                            -77.15
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------

In [7]:
import os

summary_text = []

# 각 OLS 모델 결과에서 필요한 정보를 추출하고 포맷팅합니다.
for dv_name, model_result in results.items():
    summary_text.append(f"=== {dv_name} 결과 요약 ===")
    summary_text.append(f"R-squared: {model_result.rsquared:.3f}")
    summary_text.append(f"Adj. R-squared: {model_result.rsquared_adj:.3f}") # 'adj_rsquared'를 'rsquared_adj'로 수정
    summary_text.append(f"F-statistic: {model_result.fvalue:.3f} (P-value: {model_result.f_pvalue:.3f})")
    summary_text.append("\nCoefficients:")
    summary_text.append(f"  Intercept: coef={model_result.params['Intercept']:.3f}, p>|t|={model_result.pvalues['Intercept']:.3f}")
    # C(apology_type)[T.no_apology]는 독립 변수의 두 번째 범주(no_apology)를 나타냅니다.
    # 이 예시에서는 apology_type이 defensive와 no_apology 두 가지 범주만 가진다고 가정합니다.
    # 첫 번째 범주(defensive)가 참조 범주이므로, no_apology의 계수는 defensive 대비 효과를 나타냅니다.
    try:
        summary_text.append(f"  C(apology_type)[T.no_apology]: coef={model_result.params['C(apology_type)[T.no_apology]']:.3f}, p>|t|={model_result.pvalues['C(apology_type)[T.no_apology]']:.3f}")
    except KeyError:
        # 만약 'no_apology'가 유일한 범주이거나 다른 이유로 이 계수가 없으면 처리합니다.
        summary_text.append("  C(apology_type)[T.no_apology]: (Not applicable or not found)")
    summary_text.append("\n")

# 텍스트 파일로 저장
output_filename = 'analysis_summary.txt'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_text))

print(f"분석 요약이 '{output_filename}' 파일로 저장되었습니다.")

분석 요약이 'analysis_summary.txt' 파일로 저장되었습니다.


In [9]:
# ---------------------------
# 분석 2) 프로파일 거리/유사도
# ---------------------------

neg_vars = ['Manip_ratio_neg', 'Avoid_ratio_neg', 'Excuse_ratio_neg',
            'Disrespect_ratio_neg', 'Insinc_ratio_neg']

In [10]:
# 조건별 평균 프로파일
mean_def = df[df['apology_type']=='defensive'][neg_vars].mean().values.reshape(1,-1)
mean_no = df[df['apology_type']=='no_apology'][neg_vars].mean().values.reshape(1,-1)

In [11]:
# 유클리드 거리
dist = euclidean(mean_def.flatten(), mean_no.flatten())

In [12]:
# 코사인 유사도
cos_sim = cosine_similarity(mean_def, mean_no)[0][0]

print("\n=== 프로파일 비교 ===")
print("Euclidean distance:", dist)
print("Cosine similarity:", cos_sim)


=== 프로파일 비교 ===
Euclidean distance: 0.11031534601210176
Cosine similarity: 0.9773717450747917


In [14]:
profile_summary_text = []
profile_summary_text.append("\n=== 프로파일 비교 결과 ===")
profile_summary_text.append(f"Euclidean distance: {dist:.3f}")
profile_summary_text.append(f"Cosine similarity: {cos_sim:.3f}")

# 기존 요약 파일에 프로파일 비교 결과 추가
output_filename = 'analysis_summary.txt'
with open(output_filename, 'a', encoding='utf-8') as f:
    f.write('\n'.join(profile_summary_text))

print(f"프로파일 비교 결과가 '{output_filename}' 파일에 추가되었습니다.")

프로파일 비교 결과가 'analysis_summary.txt' 파일에 추가되었습니다.
